In [5]:
import pandas as pd

# 1. Load raw weekly chart data
df = pd.read_csv('hot100.csv', low_memory=False)
df['Date'] = pd.to_datetime(df['Date'])

# 2. Compute true per-song stats directly from the raw weekly rows.
#    (The running 'Rank'/'Weeks in Charts' columns in hot100.csv are cumulative
#    snapshots-as-of-that-week, not final totals — e.g. a song's last charted
#    week usually shows it in decline, not at its true peak. So recompute
#    directly: true peak = the best (lowest) Rank it ever reached, true weeks
#    charted = how many weekly rows it has, weeks at #1 = how many of those
#    rows were Rank == 1.)
agg = df.groupby(['Song', 'Artist']).agg(
    Rank=('Rank', 'min'),
    Weeks_in_Charts=('Rank', 'size'),
    Year=('Date', lambda d: d.min().year),
).reset_index()

# 3. Sort by best peak position first, ties broken by weeks spent at #1
df_sorted = agg.sort_values(by=['Rank', 'Weeks_in_Charts'], ascending=[True, False])

# 4. Extract the stratified quotas (unchanged)
era_70s = df_sorted[(df_sorted['Year'] >= 1970) & (df_sorted['Year'] <= 1979)].head(80)
era_80s_90s = df_sorted[(df_sorted['Year'] >= 1980) & (df_sorted['Year'] <= 1999)].head(500)
era_modern = df_sorted[df_sorted['Year'] >= 2000].head(1000)

# 5. Combine and format
final_top_1000 = pd.concat([era_70s, era_80s_90s, era_modern], ignore_index=True)
final_top_1000 = final_top_1000[['Song', 'Artist', 'Year', 'Weeks_in_Charts', 'Rank']]

# Save the final dataset
final_top_1000.to_csv('stratified_top_1000_songs.csv', index=False)
print("Row-preserved database saved!")

Row-preserved database saved!


In [1]:
1+1

2